# Task 1: Rating Prediction via Prompting

## Objectives
1. Load `yelp_ratings.csv`.
2. **Sample ~250 rows** for evaluation.
3. Implement **5 Prompting Strategies**: 
    - Zero-shot Baseline
    - Few-shot
    - Chain-of-Thought
    - Strict JSON Enforcer
    - Self-Correction/Retry
4. Evaluate Accuracy, JSON Validity, and Reliability.

In [ ]:
import pandas as pd
import os
import json
from sklearn.metrics import accuracy_score

# Load Data
if os.path.exists('yelp_ratings.csv'):
    df = pd.read_csv('yelp_ratings.csv')
    print("Loaded dataset: yelp_ratings.csv")
else:
    raise FileNotFoundError("yelp_ratings.csv not found. Please upload the dataset.")

# Normalize columns
df.columns = df.columns.str.lower().str.strip()
rename_map = {
    'rating': 'stars', 'class index': 'stars',
    'review': 'text', 'review text': 'text', 'desc': 'text', 'description': 'text'
}
df.rename(columns=rename_map, inplace=True)

# --- SAMPLING STEP ---
TARGET_SAMPLE_SIZE = 250
if len(df) > TARGET_SAMPLE_SIZE:
    df = df.sample(n=TARGET_SAMPLE_SIZE, random_state=42)
    print(f"Dataset sampled to {TARGET_SAMPLE_SIZE} rows.")
    # df.to_csv('yelp_ratings.csv', index=False) # Optional save
else:
    print(f"Using full dataset ({len(df)} rows).")

print(f"Final Dataset Shape: {df.shape}")
print(df.head())

In [ ]:
def get_llm_reponse(prompt, model="gpt-3.5-turbo"):
    # Placeholder for LLM call
    # In real usage: result = openai.ChatCompletion.create(...)
    return "{\"predicted_stars\": 5, \"reasoning\": \"Positive keywords found.\"}"

In [ ]:
# Strategy 1: Zero-shot Baseline
def prompt_zero_shot(review_text):
    return f"""Classify the sentiment of this review as a star rating from 1 to 5.
Review: {review_text}
Output: JSON with 'predicted_stars'."""

In [ ]:
# Strategy 2: Few-shot
def prompt_few_shot(review_text):
    return f"""Examples:
Review: 'Loved it!' -> {{'predicted_stars': 5}}
Review: 'Terrible.' -> {{'predicted_stars': 1}}
Review: {review_text}
Output: JSON with 'predicted_stars'."""

In [ ]:
# Strategy 3: Chain-of-Thought (CoT)
def prompt_cot(review_text):
    return f"""Analyze the review step-by-step.
1. Identify positive/negative keywords.
2. Determine the tone.
3. Assign a rating (1-5).
Review: {review_text}
Format: JSON with 'reasoning' and 'predicted_stars'."""

In [ ]:
# Strategy 4: Strict JSON Enforcer
def prompt_strict_json(review_text):
    return f"""SYSTEM: You are a strict JSON data extractor.
USER: Extract the star rating (1-5) from this review.
Review: {review_text}
CRITICAL: Output ONLY valid JSON. No markdown, no preambles.
Schema: {{"predicted_stars": int, "explanation": str}}"""

In [ ]:
# Strategy 5: Self-Correction / Retry Logic
def strategy_self_correction(review_text):
    # Step 1: Initial Prompt (e.g., using Zero-shot)
    prompt = prompt_zero_shot(review_text)
    response = get_llm_reponse(prompt)
    
    # Step 2: Validation
    try:
        data = json.loads(response)
        return data # Success
    except json.JSONDecodeError:
        # Step 3: Retry/Correction Prompt
        retry_prompt = f"Previous response was invalid JSON. Fix this: {response}"
        print("Triggering self-correction...")
        return get_llm_reponse(retry_prompt) # Retry call
        
    return json.loads(response) # Fallback